# Lab 3 extension — Embedded model design space
Use the same two-session gesture CSV files from Lab 2. All comparisons use the same split. CPU only.


In [ ]:
import torch, torch.nn as nn, pandas as pd, numpy as np, time
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score
from google.colab import files
uploaded=files.upload()
raw=pd.concat([pd.read_csv(n) for n in uploaded if n.endswith('.csv')],ignore_index=True)
cols=[f'f{i}' for i in range(42)]
labels=['open_palm','fist','peace','thumbs_up']
assert set(['label','session']+cols).issubset(raw.columns)
raw=raw[raw.label.isin(labels)].dropna(subset=cols)
assert raw.session.nunique()>=2
X=raw[cols].to_numpy(dtype='float32'); y=np.array([labels.index(z) for z in raw.label])
split=GroupShuffleSplit(n_splits=1,test_size=.25,random_state=7)
a,b=next(split.split(X,y,raw.session))
assert set(raw.session.iloc[a]).isdisjoint(set(raw.session.iloc[b]))
Xtrain=torch.tensor(X[a]); ytrain=torch.tensor(y[a]); Xtest=torch.tensor(X[b]); ytest=torch.tensor(y[b])
print('Train',len(a),'Test',len(b),'Sessions',raw.session.nunique())

In [ ]:
class MLP(nn.Module):
    def __init__(self, dims):
        super().__init__()
        blocks=[]
        for i in range(len(dims)-1):
            blocks.append(nn.Linear(dims[i],dims[i+1]))
            if i<len(dims)-2: blocks.append(nn.ReLU())
        self.net=nn.Sequential(*blocks)
    def forward(self,x): return self.net(x)

def test_speed(model):
    model.eval(); x=Xtest[:1].contiguous()
    with torch.inference_mode():
        for _ in range(20):model(x)
        times=[]
        for _ in range(100):
            t=time.perf_counter();model(x);times.append((time.perf_counter()-t)*1000)
    return float(np.median(times))

def train(dims):
    torch.manual_seed(7)
    model=MLP(dims)
    optimizer=torch.optim.Adam(model.parameters(),lr=0.001)
    for epoch in range(40):
        model.train()
        for ix in torch.randperm(len(Xtrain)).split(64):
            optimizer.zero_grad();loss=nn.functional.cross_entropy(model(Xtrain[ix]),ytrain[ix]);loss.backward();optimizer.step()
    model.eval()
    with torch.inference_mode():
        acc=float((model(Xtest).argmax(1)==ytest).float().mean().item())
    count=sum(p.numel() for p in model.parameters())
    return model,{'arch':str(dims),'test_accuracy':acc,'parameters':count,'FP32_weight_KiB':count*4/1024,'CPU_batch1_median_ms':test_speed(model)}

architectures={'Tiny':[42,16,4],'Small':[42,64,32,4],'Large':[42,256,128,4]}
models={}; rows=[]
for name,dims in architectures.items():
    models[name],item=train(dims);item['model']=name;rows.append(item)
summary=pd.DataFrame(rows).set_index('model');display(summary)
summary.to_csv('gesture_design_space.csv')

## Engineering decision
Choose a model for a hypothetical resource-constrained system. Explain why it is preferred using held-out accuracy, **weight bytes**, and CPU latency. These are **not** peak SRAM, browser FPS, or MCU benchmarks.


In [ ]:
# Export one candidate. Important: the web app starter uses Small architecture by default;
# changing architecture requires updating its JavaScript model interface ONLY if shapes/class order change.
selected='Small'
chosen=models[selected].eval()
torch.onnx.export(chosen,torch.randn(1,42),'selected_design_space.onnx',
    input_names=['features'],output_names=['logits'],opset_version=17,
    dynamic_axes={'features':{0:'batch'},'logits':{0:'batch'}})
print('Saved selected_design_space.onnx')